# 14_v7: Previous-week weather baseline vs deduplicated external GIS using the same Extra Trees pipeline

## 目的

49件のstrict first-occurrence-like HPAI eventsについて，次の2モデルを
**同一の学習行，同一の負例サンプル，同一のExtra Trees設定**で比較する．

1. **Baseline model**  
   地理的位置，季節性，前週の実測気象を使用する．

2. **External GIS model**  
   Baselineに，事前に固定した解釈可能な静的GIS・土地利用特徴量のみを追加する．

## 重要な変更点

- 入力パネルを`04b`で作成した前週気象版に固定する．
- 現在週の実測気象は使用しない．
- Baselineは9変数に固定する．
- External GIS特徴量はホワイトリスト方式で選択し，自動的な列収集を行わない．
- 両モデルで同じ学習行を使用する．
- 各event weekについて，その週より前の観測だけで学習する．
- 49イベントがすべてパネル内に存在し，陽性行と一致することを検査する．
- 後続のFigure 3–5作成に利用できるcase-level，map-ready，全グリッド予測を保存する．

## Baselineの9変数

- `grid_lat`
- `grid_lon`
- `weekofyear`
- `month`
- `sin_week`
- `cos_week`
- `temp_mean_c_lag1w`
- `temp_min_c_lag1w`
- `temp_max_c_lag1w`

## v7での追加修正

`dist_to_waterbody_km`と`dist_to_lake_or_reservoir_km`は全5,491グリッドで完全一致した．
GIS生成コードでは，Natural Earth lakesから作成した距離を両列へコピーしていたため，
データ内容をより正確に表す`dist_to_lake_or_reservoir_km`だけを残し，
`dist_to_waterbody_km`はモデルから除外する．

また，採用されたGIS特徴量間の完全重複を自動検査し，
別の重複列が残っている場合はモデル学習前に停止する．


## 1. Google Drive接続，ライブラリ，保存先

In [1]:
NOTEBOOK_VERSION = (
    "14_v7_compare_baseline_external_previous_week_weather_deduplicated_gis_final"
)
print("NOTEBOOK VERSION:", NOTEBOOK_VERSION)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Google Colab以外の環境として実行します．")
except Exception as e:
    print("Drive mount warning:", repr(e))

import gc
import hashlib
import json
import platform
import re
import sys
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from scipy.stats import binomtest
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")

PROC_DIR = Path(
    "/content/drive/MyDrive/avian_influenza_project/processed"
)
RESULT_DIR = PROC_DIR / "model_outputs_riskmap_eval"
AUDIT_DIR = RESULT_DIR / "14_v7_audit"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

write_test = AUDIT_DIR / "_14_v7_drive_write_test.txt"
write_test.write_text(
    f"{NOTEBOOK_VERSION}: {datetime.now().isoformat()}\n",
    encoding="utf-8",
)

print("PROC_DIR  :", PROC_DIR)
print("RESULT_DIR:", RESULT_DIR)
print("AUDIT_DIR :", AUDIT_DIR)
print("write test:", write_test.exists())
print("Python    :", sys.version.split()[0])
print("pandas    :", pd.__version__)
print("platform  :", platform.platform())

NOTEBOOK VERSION: 14_v7_compare_baseline_external_previous_week_weather_deduplicated_gis_final
Mounted at /content/drive
PROC_DIR  : /content/drive/MyDrive/avian_influenza_project/processed
RESULT_DIR: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval
AUDIT_DIR : /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_audit
write test: True
Python    : 3.12.13
pandas    : 2.2.2
platform  : Linux-6.6.122+-x86_64-with-glibc2.35


## 2. 最終解析仕様

In [2]:
# ============================================================
# Input files
# ============================================================
PANEL_PATH = (
    PROC_DIR
    / "hpai_weekly_grid_panel_with_previous_week_weather_model_ready.parquet"
)

EVENT_CLASSIFICATION_CANDIDATES = [
    RESULT_DIR / "10_event_occurrence_type_classification.csv",
    RESULT_DIR / "10_best_model_event_cases_with_occurrence_type.csv",
    RESULT_DIR / "13_best_model_first_occurrence_cases_with_success_label.csv",
]

DOMAIN_FEATURE_CANDIDATES = [
    RESULT_DIR / "18_v8_grid_environment_features_for_14.parquet",
    RESULT_DIR / "18_v8_grid_environment_features_for_14.csv",
    PROC_DIR / "domain_features" / "grid_environment_features.parquet",
    PROC_DIR / "domain_features" / "grid_environment_features.csv",
    RESULT_DIR / "18_grid_environment_features.csv",
    RESULT_DIR / "17_grid_environment_features_for_14.parquet",
    RESULT_DIR / "17_grid_environment_features_for_14.csv",
    RESULT_DIR / "16_grid_environment_features_for_14.parquet",
    RESULT_DIR / "16_grid_environment_features_for_14.csv",
]

# ============================================================
# Expected final panel
# ============================================================
ANALYSIS_START = pd.Timestamp("2020-08-31")
ANALYSIS_END = pd.Timestamp("2026-05-18")
EXPECTED_PANEL_ROWS = 1_641_809
EXPECTED_GRID_COUNT = 5_491
EXPECTED_FIRST_EVENT_COUNT = 49

# ============================================================
# Event definition
# ============================================================
OCCURRENCE_TYPE_COLUMN = "occurrence_type_8w_30km"
EVALUATED_OCCURRENCE_TYPE = "first_occurrence_like"

# ============================================================
# Baseline: exactly 9 variables
# ============================================================
BASE_FEATURES = [
    "grid_lat",
    "grid_lon",
    "weekofyear",
    "month",
    "sin_week",
    "cos_week",
    "temp_mean_c_lag1w",
    "temp_min_c_lag1w",
    "temp_max_c_lag1w",
]

# ============================================================
# Cleaned, interpretable static GIS / land-use whitelist
# No automatically collected columns are allowed.
# ============================================================
EXCLUDED_DUPLICATE_GIS_FEATURES = {
    "dist_to_waterbody_km": (
        "Exact duplicate of dist_to_lake_or_reservoir_km across all 5,491 grids; "
        "both originated from the same Natural Earth lakes layer."
    )
}

GIS_FEATURE_WHITELIST = [
    "dist_to_coast_km",
    "dist_to_river_km",
    "dist_to_lake_or_reservoir_km",
    "waterbody_ratio_5km",
    "urban_ratio_5km",
    "elevation_m",
    "slope_deg",
    "paddy_ratio_5km",
    "farmland_ratio_5km",
    "forest_ratio_5km",
]

MIN_USABLE_GIS_FEATURES = 5

# ============================================================
# Model / sampling
# ============================================================
RANDOM_STATE = 42
N_ESTIMATORS = 250
MIN_SAMPLES_LEAF = 2
NEGATIVE_SAMPLE_RATIO = 20
MIN_NEGATIVE_SAMPLES = 1_000
MAX_NEGATIVE_SAMPLES = 300_000
TOPK_LIST = [0.01, 0.05, 0.10, 0.20, 0.30]

# Paired bootstrap
N_BOOTSTRAP = 20_000
BOOTSTRAP_RANDOM_STATE = 20260617

BASELINE_MODEL_NAME = (
    "baseline_extratrees_geo_season_previous_week_weather"
)
EXTERNAL_MODEL_NAME = (
    "external_extratrees_geo_season_previous_week_weather_gis"
)

# Current-week weather must never enter the model.
CURRENT_WEEK_WEATHER_COLUMNS = [
    "temp_mean_c",
    "temp_min_c",
    "temp_max_c",
]

FORBIDDEN_FEATURE_PATTERNS = [
    r"^y_lead",
    r"target",
    r"label",
    r"outbreak",
    r"pred",
    r"risk",
    r"rank",
    r"same_grid_past",
    r"neighbor_outbreak",
    r"lag_outbreak",
    r"rolling_outbreak",
    r"num_birds",
    r"bird_count",
    r"current_bird",
    r"week_start",
    r"weather_source_week_start",
]

print("PANEL_PATH:", PANEL_PATH)
print("BASE_FEATURES:", BASE_FEATURES)
print("GIS whitelist:", GIS_FEATURE_WHITELIST)

PANEL_PATH: /content/drive/MyDrive/avian_influenza_project/processed/hpai_weekly_grid_panel_with_previous_week_weather_model_ready.parquet
BASE_FEATURES: ['grid_lat', 'grid_lon', 'weekofyear', 'month', 'sin_week', 'cos_week', 'temp_mean_c_lag1w', 'temp_min_c_lag1w', 'temp_max_c_lag1w']
GIS whitelist: ['dist_to_coast_km', 'dist_to_river_km', 'dist_to_lake_or_reservoir_km', 'waterbody_ratio_5km', 'urban_ratio_5km', 'elevation_m', 'slope_deg', 'paddy_ratio_5km', 'farmland_ratio_5km', 'forest_ratio_5km']


## 3. 補助関数

In [3]:
def read_table(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".feather":
        return pd.read_feather(path)
    raise ValueError(f"Unsupported file format: {path}")


def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None


def require_columns(df, columns, name):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise KeyError(
            f"{name}に必要な列がありません: {missing}\n"
            f"利用可能な列: {df.columns.tolist()}"
        )


def save_csv(df, filename):
    path = RESULT_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print("Saved:", path, df.shape)
    return path


def save_audit_csv(df, filename):
    path = AUDIT_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print("Saved:", path, df.shape)
    return path


def fiscal_year_from_date(values):
    values = pd.to_datetime(values)
    return np.where(
        values.dt.month >= 4,
        values.dt.year,
        values.dt.year - 1,
    )


def canonicalize_coordinate_columns(df):
    df = df.copy()
    lat_candidates = [
        "grid_lat",
        "centroid_lat",
        "latitude",
        "lat",
    ]
    lon_candidates = [
        "grid_lon",
        "centroid_lon",
        "longitude",
        "lon",
        "lng",
    ]
    lat_col = next(
        (c for c in lat_candidates if c in df.columns),
        None,
    )
    lon_col = next(
        (c for c in lon_candidates if c in df.columns),
        None,
    )
    if lat_col is None or lon_col is None:
        raise KeyError(
            "緯度・経度列を特定できません．"
            f"columns={df.columns.tolist()}"
        )
    if lat_col != "grid_lat":
        df = df.rename(columns={lat_col: "grid_lat"})
    if lon_col != "grid_lon":
        df = df.rename(columns={lon_col: "grid_lon"})
    return df


def is_forbidden_feature(column):
    text = str(column).lower()
    return any(
        re.search(pattern, text)
        for pattern in FORBIDDEN_FEATURE_PATTERNS
    )


def make_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        ExtraTreesClassifier(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            min_samples_leaf=MIN_SAMPLES_LEAF,
            class_weight="balanced_subsample",
        ),
    )


def topk_column(k):
    return f"top{int(round(k * 100))}_hit"


def dataframe_index_hash(index_values):
    values = np.asarray(index_values, dtype=np.int64)
    return hashlib.sha256(values.tobytes()).hexdigest()


def safe_float(value):
    return float(value) if pd.notna(value) else np.nan

## 4. 前週気象モデル用パネルの読み込みと検証

`04b`の確定出力だけを読み込む．現在週気象が残っている場合は停止する．

In [4]:
if not PANEL_PATH.exists():
    raise FileNotFoundError(
        f"04bのモデル用パネルがありません: {PANEL_PATH}\n"
        "04b_build_previous_week_weather_panel_final_v2.ipynbを"
        "最初から実行してください．"
    )

panel = pd.read_parquet(PANEL_PATH)
require_columns(
    panel,
    [
        "grid_id",
        "week_start",
        "outbreak_binary",
        "weather_source_week_start",
    ]
    + BASE_FEATURES[-3:],
    "previous-week weather panel",
)

panel["grid_id"] = panel["grid_id"].astype(str)
panel["week_start"] = pd.to_datetime(
    panel["week_start"]
).dt.normalize()
panel["weather_source_week_start"] = pd.to_datetime(
    panel["weather_source_week_start"]
).dt.normalize()
panel["outbreak_binary"] = pd.to_numeric(
    panel["outbreak_binary"],
    errors="coerce",
).fillna(0).astype(int)

panel = canonicalize_coordinate_columns(panel)

# Seasonal variables are recomputed here to ensure one fixed definition.
panel["weekofyear"] = (
    panel["week_start"].dt.isocalendar().week.astype(int)
)
panel["month"] = panel["week_start"].dt.month.astype(int)
panel["sin_week"] = np.sin(
    2 * np.pi * panel["weekofyear"] / 52.1775
)
panel["cos_week"] = np.cos(
    2 * np.pi * panel["weekofyear"] / 52.1775
)

duplicate_grid_week = int(
    panel.duplicated(["grid_id", "week_start"]).sum()
)
current_weather_present = [
    c for c in CURRENT_WEEK_WEATHER_COLUMNS if c in panel.columns
]
week_gap = (
    panel["week_start"] - panel["weather_source_week_start"]
).dt.days

panel_checks = pd.DataFrame(
    [
        {
            "check": "panel file exists",
            "value": PANEL_PATH.exists(),
            "expected": True,
            "passed": PANEL_PATH.exists(),
        },
        {
            "check": "panel row count",
            "value": len(panel),
            "expected": EXPECTED_PANEL_ROWS,
            "passed": len(panel) == EXPECTED_PANEL_ROWS,
        },
        {
            "check": "grid count",
            "value": panel["grid_id"].nunique(),
            "expected": EXPECTED_GRID_COUNT,
            "passed": (
                panel["grid_id"].nunique()
                == EXPECTED_GRID_COUNT
            ),
        },
        {
            "check": "analysis start",
            "value": str(panel["week_start"].min().date()),
            "expected": str(ANALYSIS_START.date()),
            "passed": (
                panel["week_start"].min() == ANALYSIS_START
            ),
        },
        {
            "check": "analysis end",
            "value": str(panel["week_start"].max().date()),
            "expected": str(ANALYSIS_END.date()),
            "passed": panel["week_start"].max() == ANALYSIS_END,
        },
        {
            "check": "duplicate grid-week",
            "value": duplicate_grid_week,
            "expected": 0,
            "passed": duplicate_grid_week == 0,
        },
        {
            "check": "current-week weather columns absent",
            "value": current_weather_present,
            "expected": [],
            "passed": len(current_weather_present) == 0,
        },
        {
            "check": "weather source gap exactly 7 days",
            "value": sorted(week_gap.dropna().unique().tolist()),
            "expected": [7],
            "passed": (
                week_gap.notna().all()
                and week_gap.eq(7).all()
            ),
        },
        {
            "check": "previous-week weather missing values",
            "value": int(
                panel[BASE_FEATURES[-3:]].isna().sum().sum()
            ),
            "expected": 0,
            "passed": (
                panel[BASE_FEATURES[-3:]]
                .isna()
                .sum()
                .sum()
                == 0
            ),
        },
    ]
)

save_audit_csv(panel_checks, "14_v7_panel_validation.csv")
display(panel_checks)

if not panel_checks["passed"].all():
    display(panel_checks[~panel_checks["passed"]])
    raise AssertionError(
        "04bパネルの検証に失敗しました．14_v7を続行しません．"
    )

panel = panel.sort_values(
    ["week_start", "grid_id"]
).reset_index(drop=True)
panel["_row_id"] = np.arange(len(panel), dtype=np.int64)

print("panel shape:", panel.shape)
print(
    "period:",
    panel["week_start"].min(),
    "to",
    panel["week_start"].max(),
)
print("grids:", panel["grid_id"].nunique())
print("positive grid-weeks:", int(panel["outbreak_binary"].sum()))

Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_audit/14_v7_panel_validation.csv (9, 4)


,check,value,expected,passed
0,panel file exists,True,True,True
1,panel row count,1641809,1641809,True
2,grid count,5491,5491,True
3,analysis start,2020-08-31,2020-08-31,True
4,analysis end,2026-05-18,2026-05-18,True
5,duplicate grid-week,0,0,True
6,current-week weather columns absent,[],[],True
7,weather source gap exactly 7 days,[7],[7],True
8,previous-week weather missing values,0,0,True


panel shape: (1641809, 36)
period: 2020-08-31 00:00:00 to 2026-05-18 00:00:00
grids: 5491
positive grid-weeks: 191


## 5. Strict first-occurrence-like eventsの固定

8週間・30 km定義の分類列を優先し，49件でなければ停止する．
49件すべてがモデル用パネル内の陽性行に対応することも確認する．

In [5]:
EVENT_CLASSIFICATION_PATH = first_existing(
    EVENT_CLASSIFICATION_CANDIDATES
)
if EVENT_CLASSIFICATION_PATH is None:
    raise FileNotFoundError(
        "strict first-occurrence event分類ファイルがありません．\n"
        + "\n".join(
            str(p) for p in EVENT_CLASSIFICATION_CANDIDATES
        )
    )

events_raw = pd.read_csv(EVENT_CLASSIFICATION_PATH)

for column in ["week_start", "event_week"]:
    if column in events_raw.columns:
        events_raw[column] = pd.to_datetime(
            events_raw[column]
        ).dt.normalize()

if (
    "week_start" not in events_raw.columns
    and "event_week" in events_raw.columns
):
    events_raw["week_start"] = events_raw["event_week"]

if OCCURRENCE_TYPE_COLUMN in events_raw.columns:
    first_events = events_raw[
        events_raw[OCCURRENCE_TYPE_COLUMN]
        == EVALUATED_OCCURRENCE_TYPE
    ].copy()
    occurrence_source_column = OCCURRENCE_TYPE_COLUMN
elif "occurrence_type" in events_raw.columns:
    first_events = events_raw[
        events_raw["occurrence_type"]
        == EVALUATED_OCCURRENCE_TYPE
    ].copy()
    occurrence_source_column = "occurrence_type"
elif "top10_success" in events_raw.columns:
    # Fallback is accepted only if the file itself has exactly
    # the expected 49 unique event grid-weeks.
    first_events = events_raw.copy()
    occurrence_source_column = "fallback_pre_restricted_file"
else:
    raise KeyError(
        "first-occurrence-like eventを識別できません．"
        f"columns={events_raw.columns.tolist()}"
    )

require_columns(
    first_events,
    ["grid_id", "week_start"],
    "first occurrence events",
)
first_events["grid_id"] = first_events["grid_id"].astype(str)
first_events["week_start"] = pd.to_datetime(
    first_events["week_start"]
).dt.normalize()

first_events = (
    first_events[["grid_id", "week_start"]]
    .drop_duplicates()
    .sort_values(["week_start", "grid_id"])
    .reset_index(drop=True)
)
first_events["fiscal_year"] = fiscal_year_from_date(
    first_events["week_start"]
)

event_panel_check = first_events.merge(
    panel[
        [
            "grid_id",
            "week_start",
            "outbreak_binary",
            "grid_lat",
            "grid_lon",
        ]
    ],
    on=["grid_id", "week_start"],
    how="left",
    validate="one_to_one",
    indicator=True,
)
event_panel_check["matched_panel"] = (
    event_panel_check["_merge"] == "both"
)
event_panel_check["matched_positive"] = (
    event_panel_check["outbreak_binary"].fillna(0).eq(1)
)

event_checks = pd.DataFrame(
    [
        {
            "check": "strict event count",
            "value": len(first_events),
            "expected": EXPECTED_FIRST_EVENT_COUNT,
            "passed": (
                len(first_events)
                == EXPECTED_FIRST_EVENT_COUNT
            ),
        },
        {
            "check": "all events matched to panel",
            "value": int(
                event_panel_check["matched_panel"].sum()
            ),
            "expected": EXPECTED_FIRST_EVENT_COUNT,
            "passed": bool(
                event_panel_check["matched_panel"].all()
            ),
        },
        {
            "check": "all events matched to positive rows",
            "value": int(
                event_panel_check["matched_positive"].sum()
            ),
            "expected": EXPECTED_FIRST_EVENT_COUNT,
            "passed": bool(
                event_panel_check["matched_positive"].all()
            ),
        },
        {
            "check": "all event weeks in analysis period",
            "value": (
                f"{first_events['week_start'].min().date()} "
                f"to {first_events['week_start'].max().date()}"
            ),
            "expected": (
                f"{ANALYSIS_START.date()} to "
                f"{ANALYSIS_END.date()}"
            ),
            "passed": bool(
                first_events["week_start"]
                .between(
                    ANALYSIS_START,
                    ANALYSIS_END,
                    inclusive="both",
                )
                .all()
            ),
        },
    ]
)

save_csv(
    first_events,
    "14_v7_strict_first_occurrence_events_used.csv",
)
save_audit_csv(
    event_panel_check.drop(columns=["_merge"]),
    "14_v7_event_panel_match_check.csv",
)
save_audit_csv(
    event_checks,
    "14_v7_event_validation.csv",
)

print("EVENT_CLASSIFICATION_PATH:", EVENT_CLASSIFICATION_PATH)
print("occurrence source column:", occurrence_source_column)
display(event_checks)
display(
    first_events["fiscal_year"]
    .value_counts()
    .sort_index()
    .rename("events")
    .to_frame()
)

if not event_checks["passed"].all():
    display(event_panel_check.loc[
        ~event_panel_check["matched_positive"]
    ])
    raise AssertionError(
        "49件のstrict first-occurrence event固定に失敗しました．"
    )

Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_strict_first_occurrence_events_used.csv (49, 3)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_audit/14_v7_event_panel_match_check.csv (49, 8)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_audit/14_v7_event_validation.csv (4, 4)
EVENT_CLASSIFICATION_PATH: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/10_event_occurrence_type_classification.csv
occurrence source column: occurrence_type_8w_30km


,check,value,expected,passed
0,strict event count,49,49,True
1,all events matched to panel,49,49,True
2,all events matched to positive rows,49,49,True
3,all event weeks in analysis period,2023-11-20 to 2026-03-23,2020-08-31 to 2026-05-18,True


,events
fiscal_year,
2023,10
2024,21
2025,18


## 6. Static GIS・土地利用特徴量の読み込み

ホワイトリストにない列は，数値列であってもモデルへ入れない．  
存在しない列，すべて欠測の列，定数列は監査表に記録して除外する．

In [6]:
DOMAIN_FEATURE_PATH = first_existing(
    DOMAIN_FEATURE_CANDIDATES
)
if DOMAIN_FEATURE_PATH is None:
    raise FileNotFoundError(
        "grid-level GIS feature fileがありません．\n"
        + "\n".join(
            str(p) for p in DOMAIN_FEATURE_CANDIDATES
        )
    )

domain_raw = read_table(DOMAIN_FEATURE_PATH)
require_columns(domain_raw, ["grid_id"], "domain features")
domain_raw["grid_id"] = domain_raw["grid_id"].astype(str)

duplicate_domain_grid = int(
    domain_raw.duplicated("grid_id").sum()
)
if duplicate_domain_grid > 0:
    raise ValueError(
        f"GISファイルにgrid_id重複が"
        f"{duplicate_domain_grid:,}件あります．"
    )

gis_inventory_rows = []
usable_gis_features = []

for feature in GIS_FEATURE_WHITELIST:
    if feature not in domain_raw.columns:
        gis_inventory_rows.append(
            {
                "feature": feature,
                "present": False,
                "non_null": 0,
                "missing_ratio": 1.0,
                "n_unique": 0,
                "usable": False,
                "exclusion_reason": "column_not_found",
            }
        )
        continue

    numeric = pd.to_numeric(
        domain_raw[feature],
        errors="coerce",
    )
    domain_raw[feature] = numeric
    non_null = int(numeric.notna().sum())
    n_unique = int(numeric.nunique(dropna=True))
    missing_ratio = float(numeric.isna().mean())

    if non_null == 0:
        usable = False
        reason = "all_missing"
    elif n_unique <= 1:
        usable = False
        reason = "constant_or_single_value"
    else:
        usable = True
        reason = ""
        usable_gis_features.append(feature)

    gis_inventory_rows.append(
        {
            "feature": feature,
            "present": True,
            "non_null": non_null,
            "missing_ratio": missing_ratio,
            "n_unique": n_unique,
            "usable": usable,
            "exclusion_reason": reason,
        }
    )

gis_inventory = pd.DataFrame(gis_inventory_rows)
save_audit_csv(
    gis_inventory,
    "14_v7_gis_whitelist_inventory.csv",
)

if len(usable_gis_features) < MIN_USABLE_GIS_FEATURES:
    display(gis_inventory)
    raise ValueError(
        f"利用可能なGIS特徴量が{len(usable_gis_features)}列だけです．"
        f"最低{MIN_USABLE_GIS_FEATURES}列を必要とします．"
    )

domain_selected = domain_raw[
    ["grid_id"] + usable_gis_features
].copy()

panel_ext = panel.merge(
    domain_selected,
    on="grid_id",
    how="left",
    validate="many_to_one",
)

if len(panel_ext) != len(panel):
    raise AssertionError(
        "GIS結合後の行数が元パネルと一致しません．"
    )

# Re-check usability after restricting the domain file to the 5,491 model grids.
# A feature can be non-empty in the full GIS source but empty or constant in the
# final model domain.
panel_gis_rows = []
panel_usable_gis_features = []
for feature in usable_gis_features:
    non_null_rows = int(panel_ext[feature].notna().sum())
    missing_rows = int(panel_ext[feature].isna().sum())
    missing_ratio = float(panel_ext[feature].isna().mean())
    n_unique = int(panel_ext[feature].nunique(dropna=True))

    if non_null_rows == 0:
        panel_usable = False
        panel_exclusion_reason = "all_missing_on_model_grids"
    elif n_unique <= 1:
        panel_usable = False
        panel_exclusion_reason = "constant_on_model_grids"
    else:
        panel_usable = True
        panel_exclusion_reason = ""
        panel_usable_gis_features.append(feature)

    panel_gis_rows.append(
        {
            "feature": feature,
            "non_null_rows": non_null_rows,
            "missing_rows": missing_rows,
            "missing_ratio": missing_ratio,
            "n_unique": n_unique,
            "usable_on_model_grids": panel_usable,
            "exclusion_reason": panel_exclusion_reason,
        }
    )

gis_panel_coverage = pd.DataFrame(panel_gis_rows)
usable_gis_features = panel_usable_gis_features

if len(usable_gis_features) < MIN_USABLE_GIS_FEATURES:
    display(gis_panel_coverage)
    raise ValueError(
        f"5,491モデルグリッド上で利用可能なGIS特徴量が"
        f"{len(usable_gis_features)}列だけです．"
        f"最低{MIN_USABLE_GIS_FEATURES}列を必要とします．"
    )

# Exact duplicate detection among GIS features retained on the 5,491 model grids.
duplicate_pair_rows = []
for left_index, left_feature in enumerate(usable_gis_features):
    for right_feature in usable_gis_features[left_index + 1:]:
        left_values = pd.to_numeric(
            panel_ext[left_feature], errors="coerce"
        )
        right_values = pd.to_numeric(
            panel_ext[right_feature], errors="coerce"
        )

        same_missing_pattern = left_values.isna().equals(
            right_values.isna()
        )
        both_non_missing = (
            left_values.notna() & right_values.notna()
        )

        if both_non_missing.any():
            absolute_difference = (
                left_values[both_non_missing]
                - right_values[both_non_missing]
            ).abs()
            maximum_absolute_difference = float(
                absolute_difference.max()
            )
            values_equal = bool(
                np.isclose(
                    left_values[both_non_missing].to_numpy(),
                    right_values[both_non_missing].to_numpy(),
                    rtol=0.0,
                    atol=1e-12,
                    equal_nan=True,
                ).all()
            )
        else:
            maximum_absolute_difference = np.nan
            values_equal = True

        exact_duplicate = bool(
            same_missing_pattern and values_equal
        )

        duplicate_pair_rows.append(
            {
                "feature_1": left_feature,
                "feature_2": right_feature,
                "both_non_missing_rows": int(
                    both_non_missing.sum()
                ),
                "same_missing_pattern": same_missing_pattern,
                "maximum_absolute_difference": (
                    maximum_absolute_difference
                ),
                "exact_duplicate": exact_duplicate,
            }
        )

gis_duplicate_pairs = pd.DataFrame(duplicate_pair_rows)
save_audit_csv(
    gis_duplicate_pairs,
    "14_v7_gis_pairwise_duplicate_check.csv",
)

remaining_exact_duplicates = gis_duplicate_pairs[
    gis_duplicate_pairs["exact_duplicate"]
].copy()

if not remaining_exact_duplicates.empty:
    display(remaining_exact_duplicates)
    raise ValueError(
        "採用予定のGIS特徴量に完全重複列が残っています．"
    )

excluded_duplicate_features = pd.DataFrame(
    [
        {
            "excluded_feature": feature,
            "reason": reason,
        }
        for feature, reason
        in EXCLUDED_DUPLICATE_GIS_FEATURES.items()
    ]
)
save_audit_csv(
    excluded_duplicate_features,
    "14_v7_excluded_duplicate_gis_features.csv",
)

save_csv(
    pd.DataFrame(
        {
            "domain_feature_path": [
                str(DOMAIN_FEATURE_PATH)
            ]
        }
    ),
    "14_v7_domain_feature_source.csv",
)
save_csv(
    pd.DataFrame(
        {"domain_feature": usable_gis_features}
    ),
    "14_v7_domain_features_used_whitelist.csv",
)
save_audit_csv(
    gis_panel_coverage,
    "14_v7_gis_panel_coverage.csv",
)

print("DOMAIN_FEATURE_PATH:", DOMAIN_FEATURE_PATH)
print("domain shape:", domain_raw.shape)
print("usable GIS features:", usable_gis_features)
display(gis_inventory)
display(gis_panel_coverage)

Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_audit/14_v7_gis_whitelist_inventory.csv (10, 7)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_audit/14_v7_gis_pairwise_duplicate_check.csv (45, 6)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_audit/14_v7_excluded_duplicate_gis_features.csv (1, 2)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_domain_feature_source.csv (1, 1)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_domain_features_used_whitelist.csv (10, 1)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_audit/14_v7_gis_panel_coverage.csv (10, 7)
DOMAIN_FEATURE_PATH: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/18_v8_grid_environment_features_for_14

,feature,present,non_null,missing_ratio,n_unique,usable,exclusion_reason
0,dist_to_coast_km,True,5491,0.000000,5491,True,
1,dist_to_river_km,True,5491,0.000000,5491,True,
2,dist_to_lake_or_reservoir_km,True,5491,0.000000,5481,True,
3,waterbody_ratio_5km,True,4095,0.254234,3954,True,
4,urban_ratio_5km,True,4095,0.254234,3642,True,
5,elevation_m,True,5485,0.001093,1189,True,
6,slope_deg,True,5476,0.002732,5256,True,
7,paddy_ratio_5km,True,4095,0.254234,3219,True,
8,farmland_ratio_5km,True,4095,0.254234,3689,True,
9,forest_ratio_5km,True,4095,0.254234,4016,True,


,feature,non_null_rows,missing_rows,missing_ratio,n_unique,usable_on_model_grids,exclusion_reason
0,dist_to_coast_km,1641809,0,0.000000,5491,True,
1,dist_to_river_km,1641809,0,0.000000,5491,True,
2,dist_to_lake_or_reservoir_km,1641809,0,0.000000,5481,True,
3,waterbody_ratio_5km,1224405,417404,0.254234,3954,True,
4,urban_ratio_5km,1224405,417404,0.254234,3642,True,
5,elevation_m,1640015,1794,0.001093,1189,True,
6,slope_deg,1637324,4485,0.002732,5256,True,
7,paddy_ratio_5km,1224405,417404,0.254234,3219,True,
8,farmland_ratio_5km,1224405,417404,0.254234,3689,True,
9,forest_ratio_5km,1224405,417404,0.254234,4016,True,


## 7. 最終特徴量セットの固定と漏洩検査

In [7]:
require_columns(panel_ext, BASE_FEATURES, "panel baseline features")

non_numeric_base = [
    c
    for c in BASE_FEATURES
    if not pd.api.types.is_numeric_dtype(panel_ext[c])
]
if non_numeric_base:
    raise TypeError(
        f"Baselineに非数値列があります: {non_numeric_base}"
    )

feature_sets = {
    BASELINE_MODEL_NAME: BASE_FEATURES.copy(),
    EXTERNAL_MODEL_NAME: (
        BASE_FEATURES + usable_gis_features
    ),
}

feature_audit_rows = []
for model_name, features in feature_sets.items():
    for feature in features:
        feature_audit_rows.append(
            {
                "model_name": model_name,
                "feature": feature,
                "is_baseline_feature": (
                    feature in BASE_FEATURES
                ),
                "is_gis_feature": (
                    feature in usable_gis_features
                ),
                "forbidden_pattern_match": (
                    is_forbidden_feature(feature)
                ),
                "dtype": str(panel_ext[feature].dtype),
                "missing_ratio": float(
                    panel_ext[feature].isna().mean()
                ),
                "n_unique": int(
                    panel_ext[feature].nunique(dropna=True)
                ),
            }
        )

feature_audit = pd.DataFrame(feature_audit_rows)
forbidden_selected = feature_audit[
    feature_audit["forbidden_pattern_match"]
].copy()

feature_sets_df = pd.DataFrame(
    [
        {
            "model_name": model_name,
            "n_features": len(features),
            "features": ";".join(features),
        }
        for model_name, features in feature_sets.items()
    ]
)

save_csv(
    feature_sets_df,
    "14_v7_feature_sets_previous_week_weather.csv",
)
save_audit_csv(
    feature_audit,
    "14_v7_selected_feature_audit.csv",
)

print(feature_sets_df.to_string(index=False))
display(feature_audit)

if len(BASE_FEATURES) != 9:
    raise AssertionError("Baselineは9変数でなければなりません．")
if not forbidden_selected.empty:
    display(forbidden_selected)
    raise ValueError(
        "最終特徴量に漏洩禁止パターンが含まれています．"
    )
if any(
    c in feature_sets[EXTERNAL_MODEL_NAME]
    for c in CURRENT_WEEK_WEATHER_COLUMNS
):
    raise ValueError(
        "現在週気象がモデル特徴量に混入しています．"
    )

print("Final feature validation passed.")

Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_feature_sets_previous_week_weather.csv (2, 3)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_audit/14_v7_selected_feature_audit.csv (28, 8)
                                              model_name  n_features                                                                                                                                                                                                                                                                              features
    baseline_extratrees_geo_season_previous_week_weather           9                                                                                                                                                                              grid_lat;grid_lon;weekofyear;month;sin_week;cos_week;temp_mean_c_lag1w;temp_min_c_lag1w;temp_max_c_lag1w
external_extratrees

,model_name,feature,is_baseline_feature,is_gis_feature,forbidden_pattern_match,dtype,missing_ratio,n_unique
0,baseline_extratrees_geo_season_previous_week_w...,grid_lat,True,False,False,float64,0.000000,362
1,baseline_extratrees_geo_season_previous_week_w...,grid_lon,True,False,False,float64,0.000000,426
2,baseline_extratrees_geo_season_previous_week_w...,weekofyear,True,False,False,int64,0.000000,53
3,baseline_extratrees_geo_season_previous_week_w...,month,True,False,False,int64,0.000000,12
4,baseline_extratrees_geo_season_previous_week_w...,sin_week,True,False,False,float64,0.000000,53
5,baseline_extratrees_geo_season_previous_week_w...,cos_week,True,False,False,float64,0.000000,53
6,baseline_extratrees_geo_season_previous_week_w...,temp_mean_c_lag1w,True,False,False,float32,0.000000,980460
7,baseline_extratrees_geo_season_previous_week_w...,temp_min_c_lag1w,True,False,False,float32,0.000000,373388
8,baseline_extratrees_geo_season_previous_week_w...,temp_max_c_lag1w,True,False,False,float32,0.000000,271007
9,external_extratrees_geo_season_previous_week_w...,grid_lat,True,False,False,float64,0.000000,362


Final feature validation passed.


## 8. Event weekごとの共通学習サンプルを作成

各event weekについて，全陽性行を保持し，陰性行を決定論的に抽出する．  
この行集合を両モデルで共用し，モデル間で学習データが変わらないようにする．

In [8]:
event_groups = (
    first_events.groupby("week_start")["grid_id"]
    .apply(list)
    .sort_index()
)

training_index_by_week = {}
training_manifest_rows = []

for event_week in event_groups.index:
    eligible = panel_ext[
        panel_ext["week_start"] < event_week
    ]
    positive = eligible[
        eligible["outbreak_binary"] == 1
    ]
    negative = eligible[
        eligible["outbreak_binary"] == 0
    ]

    if len(positive) == 0:
        raise ValueError(
            f"{event_week.date()}より前に陽性学習例がありません．"
        )

    n_negative = min(
        len(negative),
        max(
            len(positive) * NEGATIVE_SAMPLE_RATIO,
            MIN_NEGATIVE_SAMPLES,
        ),
        MAX_NEGATIVE_SAMPLES,
    )

    # Date-specific deterministic seed.
    week_seed = (
        RANDOM_STATE
        + int(event_week.strftime("%Y%m%d"))
    ) % (2**32 - 1)

    if len(negative) > n_negative:
        sampled_negative = negative.sample(
            n=n_negative,
            random_state=week_seed,
            replace=False,
        )
    else:
        sampled_negative = negative

    training_rows = pd.concat(
        [positive, sampled_negative],
        axis=0,
    ).sample(
        frac=1.0,
        random_state=week_seed,
    )

    row_ids = training_rows["_row_id"].to_numpy(
        dtype=np.int64
    )
    training_index_by_week[event_week] = row_ids

    training_manifest_rows.append(
        {
            "event_week": event_week,
            "n_eligible_rows": len(eligible),
            "n_eligible_positive": len(positive),
            "n_eligible_negative": len(negative),
            "n_sampled_positive": len(positive),
            "n_sampled_negative": len(sampled_negative),
            "n_training_rows": len(training_rows),
            "negative_to_positive_ratio": (
                len(sampled_negative) / len(positive)
            ),
            "sampling_seed": week_seed,
            "training_row_id_sha256": (
                dataframe_index_hash(
                    np.sort(row_ids)
                )
            ),
        }
    )

training_manifest = pd.DataFrame(
    training_manifest_rows
)
save_csv(
    training_manifest,
    "14_v7_shared_training_sample_manifest.csv",
)

print("event weeks:", len(event_groups))
display(training_manifest.head())
display(training_manifest.tail())

Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_shared_training_sample_manifest.csv (35, 10)
event weeks: 35


,event_week,n_eligible_rows,n_eligible_positive,n_eligible_negative,n_sampled_positive,n_sampled_negative,n_training_rows,negative_to_positive_ratio,sampling_seed,training_row_id_sha256
0,2023-11-20,922488,130,922358,130,2600,2730,20.0,20231162,ff8ed17e60bd53111b83be2a30cabda5bb8ec2dea6c98a...
1,2023-11-27,927979,131,927848,131,2620,2751,20.0,20231169,e15ca7a1f99654ced464535bb4a9aa859dfafc4c0b655c...
2,2024-01-01,955434,134,955300,134,2680,2814,20.0,20240143,bf1828487ad7d2cbc09afa3ca640957ac49f83736abeda...
3,2024-01-22,971907,136,971771,136,2720,2856,20.0,20240164,0e5da24aad775f182a7d3b2f6f6dda755800a735063bd3...
4,2024-02-05,982889,137,982752,137,2740,2877,20.0,20240247,0403a9aaa9f5b1516bf771e178aefe42008ae4f11e0871...


,event_week,n_eligible_rows,n_eligible_positive,n_eligible_negative,n_sampled_positive,n_sampled_negative,n_training_rows,negative_to_positive_ratio,sampling_seed,training_row_id_sha256
30,2026-01-12,1537480,184,1537296,184,3680,3864,20.0,20260154,f404c4d66e8d18daa4eb4574abf280d23ff9666fe15eae...
31,2026-01-19,1542971,185,1542786,185,3700,3885,20.0,20260161,944659ce6d5e28677536229fa3b4e365d499ab2d8c63bc...
32,2026-02-16,1564935,186,1564749,186,3720,3906,20.0,20260258,ce894812ee4471138c6f1003c75288627416cfef6ddf6b...
33,2026-03-02,1575917,187,1575730,187,3740,3927,20.0,20260344,aeaf0f236fb6a2f06f5ffd18134a62bbabf9733b9afb6a...
34,2026-03-23,1592390,188,1592202,188,3760,3948,20.0,20260365,dce2cdd1be2ab51566e476234ab02c33bfb58b54587a9c...


## 9. Strict rolling evaluation

各event weekの全5,491グリッドをスコアリングし，
event gridのrank，percentile，Top-k captureを計算する．

In [9]:
case_rows = []
prediction_frames = []
training_log_rows = []
importance_rows = []

for week_number, (event_week, event_grid_ids) in enumerate(
    event_groups.items(),
    start=1,
):
    row_ids = training_index_by_week[event_week]
    training_df = panel_ext.iloc[row_ids].copy()
    scoring_df = panel_ext[
        panel_ext["week_start"] == event_week
    ].copy()

    if scoring_df.empty:
        raise ValueError(
            f"評価週の行がありません: {event_week.date()}"
        )

    n_scoring_grids = scoring_df["grid_id"].nunique()
    if n_scoring_grids != EXPECTED_GRID_COUNT:
        raise ValueError(
            f"{event_week.date()}のグリッド数が"
            f"{n_scoring_grids:,}です．"
            f"期待値={EXPECTED_GRID_COUNT:,}"
        )

    y_train = training_df[
        "outbreak_binary"
    ].astype(int)

    for model_name, feature_columns in feature_sets.items():
        model = make_model()
        model.fit(
            training_df[feature_columns],
            y_train,
        )

        model_score = model.predict_proba(
            scoring_df[feature_columns]
        )[:, 1]

        scored = scoring_df[
            [
                "grid_id",
                "week_start",
                "grid_lat",
                "grid_lon",
            ]
        ].copy()
        scored["model_name"] = model_name
        scored["model_score"] = model_score
        scored["risk_rank"] = scored[
            "model_score"
        ].rank(
            method="average",
            ascending=False,
        )
        n_grids = len(scored)
        scored["risk_percentile"] = (
            1
            - (
                scored["risk_rank"] - 1
            )
            / max(n_grids - 1, 1)
        )

        for k in TOPK_LIST:
            scored[topk_column(k)] = (
                scored["risk_rank"]
                <= np.ceil(n_grids * k)
            )

        event_scored = first_events[
            first_events["week_start"] == event_week
        ][
            ["grid_id", "week_start", "fiscal_year"]
        ].merge(
            scored,
            on=["grid_id", "week_start"],
            how="left",
            validate="one_to_one",
        )

        event_scored["matched"] = event_scored[
            "model_score"
        ].notna()
        event_scored["n_grids"] = n_grids
        event_scored["n_features"] = len(
            feature_columns
        )
        event_scored["training_row_id_sha256"] = (
            dataframe_index_hash(
                np.sort(row_ids)
            )
        )

        case_rows.extend(
            event_scored.to_dict("records")
        )
        prediction_frames.append(scored)

        estimator = model.named_steps[
            "extratreesclassifier"
        ]
        for feature, importance in zip(
            feature_columns,
            estimator.feature_importances_,
        ):
            importance_rows.append(
                {
                    "event_week": event_week,
                    "model_name": model_name,
                    "feature": feature,
                    "importance": float(importance),
                }
            )

        training_log_rows.append(
            {
                "event_week": event_week,
                "model_name": model_name,
                "status": "ok",
                "n_train": len(training_df),
                "n_train_positive": int(
                    y_train.sum()
                ),
                "n_train_negative": int(
                    (y_train == 0).sum()
                ),
                "n_features": len(feature_columns),
                "n_scoring_grids": n_grids,
                "n_event_grids": len(event_grid_ids),
                "n_events_matched": int(
                    event_scored["matched"].sum()
                ),
                "training_row_id_sha256": (
                    dataframe_index_hash(
                        np.sort(row_ids)
                    )
                ),
            }
        )

        del model, scored, event_scored
        gc.collect()

    print(
        f"{week_number:02d}/{len(event_groups):02d}",
        event_week.date(),
        "events=",
        len(event_grid_ids),
        "training rows=",
        len(training_df),
    )

case_results = pd.DataFrame(case_rows)
all_grid_predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
)
training_log = pd.DataFrame(training_log_rows)
feature_importance_long = pd.DataFrame(importance_rows)

case_results["week_start"] = pd.to_datetime(
    case_results["week_start"]
)
all_grid_predictions["week_start"] = pd.to_datetime(
    all_grid_predictions["week_start"]
)
training_log["event_week"] = pd.to_datetime(
    training_log["event_week"]
)
feature_importance_long["event_week"] = pd.to_datetime(
    feature_importance_long["event_week"]
)

save_csv(
    case_results,
    "14_v7_case_results_previous_week_weather_same_extratrees.csv",
)
save_csv(
    training_log,
    "14_v7_training_log_previous_week_weather_same_extratrees.csv",
)
save_csv(
    feature_importance_long,
    "14_v7_feature_importance_by_event_week.csv",
)

prediction_path = (
    RESULT_DIR
    / "14_v7_all_grid_predictions_event_weeks.parquet"
)
all_grid_predictions.to_parquet(
    prediction_path,
    index=False,
)
print(
    "Saved:",
    prediction_path,
    all_grid_predictions.shape,
)

save_csv(
    all_grid_predictions.head(20_000),
    "14_v7_all_grid_predictions_event_weeks_sample.csv",
)

print("case_results:", case_results.shape)
print(
    "all_grid_predictions:",
    all_grid_predictions.shape,
)
display(case_results.head())

01/35 2023-11-20 events= 1 training rows= 2730
02/35 2023-11-27 events= 3 training rows= 2751
03/35 2024-01-01 events= 2 training rows= 2814
04/35 2024-01-22 events= 1 training rows= 2856
05/35 2024-02-05 events= 2 training rows= 2877
06/35 2024-03-11 events= 1 training rows= 2919
07/35 2024-04-29 events= 1 training rows= 2940
08/35 2024-10-14 events= 1 training rows= 2961
09/35 2024-10-21 events= 2 training rows= 2982
10/35 2024-10-28 events= 1 training rows= 3024
11/35 2024-11-04 events= 3 training rows= 3045
12/35 2024-11-11 events= 1 training rows= 3108
13/35 2024-11-18 events= 2 training rows= 3129
14/35 2024-11-25 events= 1 training rows= 3171
15/35 2024-12-02 events= 1 training rows= 3192
16/35 2024-12-09 events= 1 training rows= 3213
17/35 2024-12-16 events= 1 training rows= 3234
18/35 2024-12-23 events= 1 training rows= 3276
19/35 2024-12-30 events= 3 training rows= 3297
20/35 2025-01-06 events= 1 training rows= 3360
21/35 2025-01-27 events= 1 training rows= 3528
22/35 2025-10

,grid_id,week_start,fiscal_year,grid_lat,grid_lon,model_name,model_score,risk_rank,risk_percentile,top1_hit,top5_hit,top10_hit,top20_hit,top30_hit,matched,n_grids,n_features,training_row_id_sha256
0,G000416,2023-11-20,2023,33.065247,130.074213,baseline_extratrees_geo_season_previous_week_w...,0.454963,596.0,0.891621,False,False,False,True,True,True,5491,9,ff8ed17e60bd53111b83be2a30cabda5bb8ec2dea6c98a...
1,G000416,2023-11-20,2023,33.065247,130.074213,external_extratrees_geo_season_previous_week_w...,0.395503,418.0,0.924044,False,False,True,True,True,True,5491,19,ff8ed17e60bd53111b83be2a30cabda5bb8ec2dea6c98a...
2,G000508,2023-11-27,2023,32.081150,130.343707,baseline_extratrees_geo_season_previous_week_w...,0.719418,73.0,0.986885,False,True,True,True,True,True,5491,9,e15ca7a1f99654ced464535bb4a9aa859dfafc4c0b655c...
3,G003353,2023-11-27,2023,35.951629,139.326860,baseline_extratrees_geo_season_previous_week_w...,0.517376,732.0,0.866849,False,False,False,True,True,True,5491,9,e15ca7a1f99654ced464535bb4a9aa859dfafc4c0b655c...
4,G004012,2023-11-27,2023,36.386740,140.225176,baseline_extratrees_geo_season_previous_week_w...,0.673475,122.0,0.977960,False,True,True,True,True,True,5491,9,e15ca7a1f99654ced464535bb4a9aa859dfafc4c0b655c...


## 10. モデル別・年度別の集計

In [10]:
def summarize_cases(df, group_columns):
    summary_rows = []
    grouped = df.groupby(
        group_columns,
        dropna=False,
    )

    for keys, group in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {
            column: value
            for column, value in zip(
                group_columns,
                keys,
            )
        }

        matched = group[group["matched"]].copy()
        row["events"] = len(group)
        row["matched_events"] = len(matched)

        for k in TOPK_LIST:
            column = topk_column(k)
            prefix = f"top{int(round(k * 100))}"
            row[f"{prefix}_events"] = int(
                matched[column].sum()
            )
            row[f"{prefix}_capture_rate"] = float(
                matched[column].mean()
            )

        row["mean_risk_percentile"] = float(
            matched["risk_percentile"].mean()
        )
        row["median_risk_percentile"] = float(
            matched["risk_percentile"].median()
        )
        row["mean_rank"] = float(
            matched["risk_rank"].mean()
        )
        row["median_rank"] = float(
            matched["risk_rank"].median()
        )
        row["mean_model_score"] = float(
            matched["model_score"].mean()
        )
        row["median_model_score"] = float(
            matched["model_score"].median()
        )
        summary_rows.append(row)

    return pd.DataFrame(summary_rows)


summary_by_model = summarize_cases(
    case_results,
    ["model_name"],
)
summary_by_fiscal_year_model = summarize_cases(
    case_results,
    ["fiscal_year", "model_name"],
)
summary_by_week_model = summarize_cases(
    case_results,
    ["week_start", "model_name"],
)

save_csv(
    summary_by_model,
    "14_v7_summary_by_model_previous_week_weather.csv",
)
save_csv(
    summary_by_fiscal_year_model,
    "14_v7_summary_by_fiscal_year_model_previous_week_weather.csv",
)
save_csv(
    summary_by_week_model,
    "14_v7_summary_by_week_model_previous_week_weather.csv",
)

display(summary_by_model)
display(summary_by_fiscal_year_model)

Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_summary_by_model_previous_week_weather.csv (2, 19)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_summary_by_fiscal_year_model_previous_week_weather.csv (6, 20)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_summary_by_week_model_previous_week_weather.csv (70, 20)


,model_name,events,matched_events,top1_events,top1_capture_rate,top5_events,top5_capture_rate,top10_events,top10_capture_rate,top20_events,top20_capture_rate,top30_events,top30_capture_rate,mean_risk_percentile,median_risk_percentile,mean_rank,median_rank,mean_model_score,median_model_score
0,baseline_extratrees_geo_season_previous_week_w...,49,49,3,0.061224,10,0.204082,11,0.224490,21,0.428571,24,0.489796,0.677936,0.696175,1769.132653,1669.0,0.364431,0.327672
1,external_extratrees_geo_season_previous_week_w...,49,49,6,0.122449,13,0.265306,19,0.387755,29,0.591837,34,0.693878,0.757832,0.855373,1330.500000,795.0,0.320190,0.243143


,fiscal_year,model_name,events,matched_events,top1_events,top1_capture_rate,top5_events,top5_capture_rate,top10_events,top10_capture_rate,top20_events,top20_capture_rate,top30_events,top30_capture_rate,mean_risk_percentile,median_risk_percentile,mean_rank,median_rank,mean_model_score,median_model_score
0,2023,baseline_extratrees_geo_season_previous_week_w...,10,10,0,0.000000,2,0.200000,2,0.200000,6,0.600000,6,0.600000,0.735055,0.821494,1455.550000,981.0,0.366102,0.314955
1,2023,external_extratrees_geo_season_previous_week_w...,10,10,1,0.100000,2,0.200000,4,0.400000,5,0.500000,5,0.500000,0.731457,0.779690,1475.300000,1210.5,0.294981,0.256787
2,2024,baseline_extratrees_geo_season_previous_week_w...,21,21,3,0.142857,7,0.333333,7,0.333333,9,0.428571,10,0.476190,0.652515,0.696175,1908.690476,1669.0,0.387042,0.391653
3,2024,external_extratrees_geo_season_previous_week_w...,21,21,5,0.238095,8,0.380952,10,0.476190,13,0.619048,14,0.666667,0.766177,0.892350,1284.690476,592.0,0.364539,0.243143
4,2025,baseline_extratrees_geo_season_previous_week_w...,18,18,0,0.000000,1,0.055556,2,0.111111,6,0.333333,8,0.444444,0.675860,0.640528,1780.527778,1974.5,0.337125,0.339144
5,2025,external_extratrees_geo_season_previous_week_w...,18,18,0,0.000000,3,0.166667,5,0.277778,11,0.611111,15,0.833333,0.762750,0.848998,1303.500000,830.0,0.282455,0.268705


## 11. Event-level paired comparison

External GISによるpercentile変化，rank改善，Top 10遷移を同一49件で算出する．

In [11]:
baseline_cases = case_results[
    case_results["model_name"]
    == BASELINE_MODEL_NAME
].copy()
external_cases = case_results[
    case_results["model_name"]
    == EXTERNAL_MODEL_NAME
].copy()

comparison = baseline_cases.merge(
    external_cases,
    on=["grid_id", "week_start"],
    how="outer",
    suffixes=("_baseline", "_external"),
    validate="one_to_one",
)

comparison["week_start"] = pd.to_datetime(
    comparison["week_start"]
)
comparison["fiscal_year"] = fiscal_year_from_date(
    comparison["week_start"]
)
comparison["matched_both"] = (
    comparison["matched_baseline"].fillna(False)
    & comparison["matched_external"].fillna(False)
)

comparison["delta_risk_percentile"] = (
    comparison["risk_percentile_external"]
    - comparison["risk_percentile_baseline"]
)
comparison["rank_improvement"] = (
    comparison["risk_rank_baseline"]
    - comparison["risk_rank_external"]
)

comparison["baseline_top10"] = comparison[
    "top10_hit_baseline"
].fillna(False).astype(bool)
comparison["external_top10"] = comparison[
    "top10_hit_external"
].fillna(False).astype(bool)

transition_conditions = [
    (
        ~comparison["baseline_top10"]
        & comparison["external_top10"]
    ),
    (
        comparison["baseline_top10"]
        & ~comparison["external_top10"]
    ),
    (
        comparison["baseline_top10"]
        & comparison["external_top10"]
    ),
    (
        ~comparison["baseline_top10"]
        & ~comparison["external_top10"]
    ),
]
transition_labels = [
    "Gain to Top 10",
    "Loss from Top 10",
    "Stable Top 10",
    "Stable miss",
]
comparison["top10_transition"] = np.select(
    transition_conditions,
    transition_labels,
    default="Unclassified",
)

if (
    len(comparison) != EXPECTED_FIRST_EVENT_COUNT
    or not comparison["matched_both"].all()
):
    raise AssertionError(
        "両モデルで同じ49件を比較できていません．"
    )

save_csv(
    comparison,
    "14_v7_case_level_baseline_vs_external_previous_week_weather.csv",
)

transition_summary = (
    comparison.groupby(
        "top10_transition",
        as_index=False,
    )
    .agg(
        events=("grid_id", "count"),
        mean_baseline_percentile=(
            "risk_percentile_baseline",
            "mean",
        ),
        mean_external_percentile=(
            "risk_percentile_external",
            "mean",
        ),
        mean_delta_percentile=(
            "delta_risk_percentile",
            "mean",
        ),
        median_delta_percentile=(
            "delta_risk_percentile",
            "median",
        ),
        mean_rank_improvement=(
            "rank_improvement",
            "mean",
        ),
        median_rank_improvement=(
            "rank_improvement",
            "median",
        ),
    )
)

fiscal_year_summary = (
    comparison.groupby(
        "fiscal_year",
        as_index=False,
    )
    .agg(
        events=("grid_id", "count"),
        gain_to_top10=(
            "top10_transition",
            lambda values: int(
                (values == "Gain to Top 10").sum()
            ),
        ),
        loss_from_top10=(
            "top10_transition",
            lambda values: int(
                (values == "Loss from Top 10").sum()
            ),
        ),
        stable_top10=(
            "top10_transition",
            lambda values: int(
                (values == "Stable Top 10").sum()
            ),
        ),
        stable_miss=(
            "top10_transition",
            lambda values: int(
                (values == "Stable miss").sum()
            ),
        ),
        baseline_top10_rate=(
            "baseline_top10",
            "mean",
        ),
        external_top10_rate=(
            "external_top10",
            "mean",
        ),
        mean_delta_percentile=(
            "delta_risk_percentile",
            "mean",
        ),
    )
)

overall_rows = []
for k in TOPK_LIST:
    pct = int(round(k * 100))
    baseline_column = f"top{pct}_hit_baseline"
    external_column = f"top{pct}_hit_external"
    base_rate = float(
        comparison[baseline_column].mean()
    )
    external_rate = float(
        comparison[external_column].mean()
    )
    overall_rows.append(
        {
            "metric": f"Top {pct}% capture rate",
            "baseline_value": base_rate,
            "external_value": external_rate,
            "absolute_change": (
                external_rate - base_rate
            ),
            "unit": "proportion",
        }
    )

overall_rows.extend(
    [
        {
            "metric": "Mean risk percentile",
            "baseline_value": float(
                comparison[
                    "risk_percentile_baseline"
                ].mean()
            ),
            "external_value": float(
                comparison[
                    "risk_percentile_external"
                ].mean()
            ),
            "absolute_change": float(
                comparison[
                    "delta_risk_percentile"
                ].mean()
            ),
            "unit": "percentile",
        },
        {
            "metric": "Median risk percentile",
            "baseline_value": float(
                comparison[
                    "risk_percentile_baseline"
                ].median()
            ),
            "external_value": float(
                comparison[
                    "risk_percentile_external"
                ].median()
            ),
            "absolute_change": float(
                comparison[
                    "risk_percentile_external"
                ].median()
                - comparison[
                    "risk_percentile_baseline"
                ].median()
            ),
            "unit": "percentile",
        },
    ]
)
overall_comparison = pd.DataFrame(overall_rows)

save_csv(
    overall_comparison,
    "14_v7_overall_baseline_vs_external_previous_week_weather.csv",
)
save_csv(
    transition_summary,
    "14_v7_gain_loss_category_summary_previous_week_weather.csv",
)
save_csv(
    fiscal_year_summary,
    "14_v7_gain_loss_summary_by_fiscal_year_previous_week_weather.csv",
)

display(overall_comparison)
display(transition_summary)
display(fiscal_year_summary)

Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_case_level_baseline_vs_external_previous_week_weather.csv (49, 41)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_overall_baseline_vs_external_previous_week_weather.csv (7, 5)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_gain_loss_category_summary_previous_week_weather.csv (4, 8)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_gain_loss_summary_by_fiscal_year_previous_week_weather.csv (3, 9)


,metric,baseline_value,external_value,absolute_change,unit
0,Top 1% capture rate,0.061224,0.122449,0.061224,proportion
1,Top 5% capture rate,0.204082,0.265306,0.061224,proportion
2,Top 10% capture rate,0.224490,0.387755,0.163265,proportion
3,Top 20% capture rate,0.428571,0.591837,0.163265,proportion
4,Top 30% capture rate,0.489796,0.693878,0.204082,proportion
5,Mean risk percentile,0.677936,0.757832,0.079897,percentile
6,Median risk percentile,0.696175,0.855373,0.159199,percentile


,top10_transition,events,mean_baseline_percentile,mean_external_percentile,mean_delta_percentile,median_delta_percentile,mean_rank_improvement,median_rank_improvement
0,Gain to Top 10,12,0.674848,0.948345,0.273497,0.199454,1501.500000,1095.0
1,Loss from Top 10,4,0.966120,0.850000,-0.116120,-0.109290,-637.500000,-600.0
2,Stable Top 10,7,0.980536,0.993547,0.013011,0.000729,71.428571,4.0
3,Stable miss,26,0.553555,0.592262,0.038707,0.089526,212.500000,491.5


,fiscal_year,events,gain_to_top10,loss_from_top10,stable_top10,stable_miss,baseline_top10_rate,external_top10_rate,mean_delta_percentile
0,2023,10,3,1,1,5,0.200000,0.400000,-0.003597
1,2024,21,4,1,6,10,0.333333,0.476190,0.113661
2,2025,18,5,2,0,11,0.111111,0.277778,0.086890


## 12. Paired bootstrap confidence intervalsとTop 10遷移のexact test

49イベントを単位としてペアを維持したbootstrapを行う．  
Top 10のgainとlossについては，discordant pairに対するexact binomial testを出力する．

In [12]:
rng = np.random.default_rng(BOOTSTRAP_RANDOM_STATE)
n_events = len(comparison)
bootstrap_rows = []

baseline_top10_array = comparison[
    "baseline_top10"
].astype(float).to_numpy()
external_top10_array = comparison[
    "external_top10"
].astype(float).to_numpy()
delta_percentile_array = comparison[
    "delta_risk_percentile"
].to_numpy(dtype=float)

for _ in range(N_BOOTSTRAP):
    indices = rng.integers(
        0,
        n_events,
        size=n_events,
    )
    bootstrap_rows.append(
        {
            "top10_capture_rate_difference": float(
                external_top10_array[indices].mean()
                - baseline_top10_array[indices].mean()
            ),
            "mean_delta_risk_percentile": float(
                delta_percentile_array[indices].mean()
            ),
            "median_delta_risk_percentile": float(
                np.median(
                    delta_percentile_array[indices]
                )
            ),
        }
    )

bootstrap_results = pd.DataFrame(bootstrap_rows)

bootstrap_summary_rows = []
observed_statistics = {
    "top10_capture_rate_difference": float(
        external_top10_array.mean()
        - baseline_top10_array.mean()
    ),
    "mean_delta_risk_percentile": float(
        delta_percentile_array.mean()
    ),
    "median_delta_risk_percentile": float(
        np.median(delta_percentile_array)
    ),
}

for statistic, observed in observed_statistics.items():
    values = bootstrap_results[statistic]
    bootstrap_summary_rows.append(
        {
            "statistic": statistic,
            "observed": observed,
            "bootstrap_mean": float(values.mean()),
            "ci_2_5": float(values.quantile(0.025)),
            "ci_97_5": float(values.quantile(0.975)),
            "n_bootstrap": N_BOOTSTRAP,
        }
    )

bootstrap_summary = pd.DataFrame(
    bootstrap_summary_rows
)

n_gain = int(
    (
        comparison["top10_transition"]
        == "Gain to Top 10"
    ).sum()
)
n_loss = int(
    (
        comparison["top10_transition"]
        == "Loss from Top 10"
    ).sum()
)
n_discordant = n_gain + n_loss

if n_discordant > 0:
    exact_result = binomtest(
        k=n_gain,
        n=n_discordant,
        p=0.5,
        alternative="two-sided",
    )
    exact_p = float(exact_result.pvalue)
else:
    exact_p = np.nan

top10_exact_test = pd.DataFrame(
    [
        {
            "test": (
                "Exact paired discordant-transition "
                "binomial test"
            ),
            "gain_to_top10": n_gain,
            "loss_from_top10": n_loss,
            "discordant_pairs": n_discordant,
            "two_sided_p_value": exact_p,
        }
    ]
)

save_csv(
    bootstrap_summary,
    "14_v7_paired_bootstrap_summary.csv",
)
save_csv(
    top10_exact_test,
    "14_v7_top10_transition_exact_test.csv",
)
save_audit_csv(
    bootstrap_results,
    "14_v7_paired_bootstrap_replicates.csv",
)

display(bootstrap_summary)
display(top10_exact_test)

Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_paired_bootstrap_summary.csv (3, 6)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_top10_transition_exact_test.csv (1, 5)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_audit/14_v7_paired_bootstrap_replicates.csv (20000, 3)


,statistic,observed,bootstrap_mean,ci_2_5,ci_97_5,n_bootstrap
0,top10_capture_rate_difference,0.163265,0.162419,0.020408,0.306122,20000
1,mean_delta_risk_percentile,0.079897,0.079623,0.019406,0.141986,20000
2,median_delta_risk_percentile,0.071038,0.065988,0.001457,0.138616,20000


,test,gain_to_top10,loss_from_top10,discordant_pairs,two_sided_p_value
0,Exact paired discordant-transition binomial test,12,4,16,0.076813


## 13. Figure・Table作成用の派生ファイル

- baselineのTop 10 success / miss
- external GISのTop 10 success / miss
- gain / loss / stable分類のmap-readyデータ
- GIS変数を付加したevent-levelデータ

In [13]:
map_ready = comparison[
    [
        "grid_id",
        "week_start",
        "fiscal_year",
        "grid_lat_baseline",
        "grid_lon_baseline",
        "risk_rank_baseline",
        "risk_percentile_baseline",
        "risk_rank_external",
        "risk_percentile_external",
        "delta_risk_percentile",
        "rank_improvement",
        "baseline_top10",
        "external_top10",
        "top10_transition",
    ]
].rename(
    columns={
        "grid_lat_baseline": "grid_lat",
        "grid_lon_baseline": "grid_lon",
    }
)

event_gis = panel_ext[
    ["grid_id"] + usable_gis_features
].drop_duplicates("grid_id")

enriched_comparison = comparison.merge(
    event_gis,
    on="grid_id",
    how="left",
    validate="many_to_one",
)

baseline_success_miss = baseline_cases[
    [
        "grid_id",
        "week_start",
        "fiscal_year",
        "grid_lat",
        "grid_lon",
        "model_score",
        "risk_rank",
        "risk_percentile",
        "top1_hit",
        "top5_hit",
        "top10_hit",
        "top20_hit",
        "top30_hit",
    ]
].copy()
baseline_success_miss["top10_group"] = np.where(
    baseline_success_miss["top10_hit"],
    "Top 10 success",
    "Top 10 miss",
)

external_success_miss = external_cases[
    [
        "grid_id",
        "week_start",
        "fiscal_year",
        "grid_lat",
        "grid_lon",
        "model_score",
        "risk_rank",
        "risk_percentile",
        "top1_hit",
        "top5_hit",
        "top10_hit",
        "top20_hit",
        "top30_hit",
    ]
].copy()
external_success_miss["top10_group"] = np.where(
    external_success_miss["top10_hit"],
    "Top 10 success",
    "Top 10 miss",
)

feature_summary_by_transition = (
    enriched_comparison.groupby(
        "top10_transition",
        as_index=False,
    )[usable_gis_features]
    .mean(numeric_only=True)
)

save_csv(
    map_ready,
    "14_v7_map_ready_external_gis_gain_loss_cases.csv",
)
save_csv(
    enriched_comparison,
    "14_v7_case_level_gain_loss_enriched.csv",
)
save_csv(
    baseline_success_miss,
    "14_v7_baseline_top10_success_miss_cases.csv",
)
save_csv(
    external_success_miss,
    "14_v7_external_top10_success_miss_cases.csv",
)
save_csv(
    feature_summary_by_transition,
    "14_v7_gis_feature_summary_by_gain_loss_category.csv",
)

display(
    baseline_success_miss[
        "top10_group"
    ].value_counts()
)
display(map_ready["top10_transition"].value_counts())
display(feature_summary_by_transition)

Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_map_ready_external_gis_gain_loss_cases.csv (49, 14)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_case_level_gain_loss_enriched.csv (49, 51)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_baseline_top10_success_miss_cases.csv (49, 14)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_external_top10_success_miss_cases.csv (49, 14)
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_gis_feature_summary_by_gain_loss_category.csv (4, 11)


,count
top10_group,
Top 10 miss,38
Top 10 success,11


,count
top10_transition,
Stable miss,26
Gain to Top 10,12
Stable Top 10,7
Loss from Top 10,4


,top10_transition,dist_to_coast_km,dist_to_river_km,dist_to_lake_or_reservoir_km,waterbody_ratio_5km,urban_ratio_5km,elevation_m,slope_deg,paddy_ratio_5km,farmland_ratio_5km,forest_ratio_5km
0,Gain to Top 10,31.963771,186.784921,436.752655,0.080292,0.149283,82.083333,1.487164,0.236107,0.133015,0.401303
1,Loss from Top 10,19.953605,208.590261,309.378114,0.090759,0.119009,55.250000,1.287372,0.223269,0.080642,0.486320
2,Stable Top 10,8.885421,355.115329,656.317416,0.041414,0.149406,57.142857,2.643931,0.210054,0.224924,0.374202
3,Stable miss,38.407964,266.087438,538.907752,0.044631,0.073773,216.346154,3.023123,0.111386,0.078711,0.691499


## 14. Feature importanceの集約

In [14]:
feature_importance_summary = (
    feature_importance_long.groupby(
        ["model_name", "feature"],
        as_index=False,
    )
    .agg(
        mean_importance=("importance", "mean"),
        median_importance=("importance", "median"),
        min_importance=("importance", "min"),
        max_importance=("importance", "max"),
        event_weeks=("event_week", "nunique"),
    )
    .sort_values(
        ["model_name", "mean_importance"],
        ascending=[True, False],
    )
)

save_csv(
    feature_importance_summary,
    "14_v7_feature_importance_summary.csv",
)
display(
    feature_importance_summary.groupby(
        "model_name",
        group_keys=False,
    ).head(20)
)

Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_feature_importance_summary.csv (28, 7)


,model_name,feature,mean_importance,median_importance,min_importance,max_importance,event_weeks
0,baseline_extratrees_geo_season_previous_week_w...,cos_week,0.236633,0.237049,0.215384,0.264052,35
1,baseline_extratrees_geo_season_previous_week_w...,grid_lat,0.126117,0.125251,0.111938,0.135988,35
6,baseline_extratrees_geo_season_previous_week_w...,temp_mean_c_lag1w,0.104013,0.104156,0.098011,0.109757,35
5,baseline_extratrees_geo_season_previous_week_w...,temp_max_c_lag1w,0.103833,0.103744,0.096396,0.111839,35
7,baseline_extratrees_geo_season_previous_week_w...,temp_min_c_lag1w,0.099678,0.099230,0.088988,0.108942,35
8,baseline_extratrees_geo_season_previous_week_w...,weekofyear,0.093282,0.092874,0.086299,0.101001,35
2,baseline_extratrees_geo_season_previous_week_w...,grid_lon,0.091169,0.091641,0.081449,0.098707,35
4,baseline_extratrees_geo_season_previous_week_w...,sin_week,0.074539,0.074673,0.070534,0.080120,35
3,baseline_extratrees_geo_season_previous_week_w...,month,0.070736,0.070826,0.061659,0.077905,35
9,external_extratrees_geo_season_previous_week_w...,cos_week,0.178995,0.177163,0.162843,0.204655,35


## 15. 最終整合性検査

数値を論文へ移す前に，49件，両モデルの共通学習行，予測グリッド数，
Top 10遷移の合計を検査する．

In [15]:
training_hash_check = (
    training_log.pivot(
        index="event_week",
        columns="model_name",
        values="training_row_id_sha256",
    )
)
same_training_hash_all_weeks = bool(
    (
        training_hash_check[BASELINE_MODEL_NAME]
        == training_hash_check[EXTERNAL_MODEL_NAME]
    ).all()
)

transition_total = int(
    transition_summary["events"].sum()
)
prediction_grid_counts = (
    all_grid_predictions.groupby(
        ["week_start", "model_name"]
    )["grid_id"]
    .nunique()
)

final_checks = pd.DataFrame(
    [
        {
            "check": "case rows per model",
            "value": (
                case_results.groupby("model_name")
                .size()
                .to_dict()
            ),
            "expected": (
                f"{EXPECTED_FIRST_EVENT_COUNT} per model"
            ),
            "passed": bool(
                case_results.groupby("model_name")
                .size()
                .eq(EXPECTED_FIRST_EVENT_COUNT)
                .all()
            ),
        },
        {
            "check": "all cases matched",
            "value": int(case_results["matched"].sum()),
            "expected": (
                EXPECTED_FIRST_EVENT_COUNT * 2
            ),
            "passed": bool(
                case_results["matched"].all()
            ),
        },
        {
            "check": "same training rows for both models",
            "value": same_training_hash_all_weeks,
            "expected": True,
            "passed": same_training_hash_all_weeks,
        },
        {
            "check": "all scored weeks contain 5491 grids",
            "value": sorted(
                prediction_grid_counts.unique().tolist()
            ),
            "expected": [EXPECTED_GRID_COUNT],
            "passed": bool(
                prediction_grid_counts.eq(
                    EXPECTED_GRID_COUNT
                ).all()
            ),
        },
        {
            "check": "Top 10 transition total",
            "value": transition_total,
            "expected": EXPECTED_FIRST_EVENT_COUNT,
            "passed": (
                transition_total
                == EXPECTED_FIRST_EVENT_COUNT
            ),
        },
        {
            "check": "baseline feature count",
            "value": len(BASE_FEATURES),
            "expected": 9,
            "passed": len(BASE_FEATURES) == 9,
        },
        {
            "check": "known duplicate waterbody feature excluded",
            "value": (
                "dist_to_waterbody_km"
                in feature_sets[EXTERNAL_MODEL_NAME]
            ),
            "expected": False,
            "passed": (
                "dist_to_waterbody_km"
                not in feature_sets[EXTERNAL_MODEL_NAME]
            ),
        },
        {
            "check": "lake or reservoir distance retained",
            "value": (
                "dist_to_lake_or_reservoir_km"
                in feature_sets[EXTERNAL_MODEL_NAME]
            ),
            "expected": True,
            "passed": (
                "dist_to_lake_or_reservoir_km"
                in feature_sets[EXTERNAL_MODEL_NAME]
            ),
        },
        {
            "check": "no exact duplicate GIS feature pairs remain",
            "value": len(remaining_exact_duplicates),
            "expected": 0,
            "passed": len(remaining_exact_duplicates) == 0,
        },
        {
            "check": "external features are baseline plus whitelist",
            "value": len(
                feature_sets[EXTERNAL_MODEL_NAME]
            ),
            "expected": (
                len(BASE_FEATURES)
                + len(usable_gis_features)
            ),
            "passed": (
                feature_sets[EXTERNAL_MODEL_NAME]
                == BASE_FEATURES + usable_gis_features
            ),
        },
        {
            "check": "current-week weather absent",
            "value": [
                c
                for c in CURRENT_WEEK_WEATHER_COLUMNS
                if any(
                    c in features
                    for features in feature_sets.values()
                )
            ],
            "expected": [],
            "passed": not any(
                c in features
                for features in feature_sets.values()
                for c in CURRENT_WEEK_WEATHER_COLUMNS
            ),
        },
    ]
)

save_audit_csv(
    final_checks,
    "14_v7_final_validation_checks.csv",
)
display(final_checks)

if not final_checks["passed"].all():
    display(final_checks[~final_checks["passed"]])
    raise AssertionError(
        "14_v7の最終整合性検査に失敗しました．"
    )

print("All 14_v7 final checks passed.")

Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_audit/14_v7_final_validation_checks.csv (11, 4)


,check,value,expected,passed
0,case rows per model,{'baseline_extratrees_geo_season_previous_week...,49 per model,True
1,all cases matched,98,98,True
2,same training rows for both models,True,True,True
3,all scored weeks contain 5491 grids,[5491],[5491],True
4,Top 10 transition total,49,49,True
5,baseline feature count,9,9,True
6,known duplicate waterbody feature excluded,False,False,True
7,lake or reservoir distance retained,True,True,True
8,no exact duplicate GIS feature pairs remain,0,0,True
9,external features are baseline plus whitelist,19,19,True


All 14_v7 final checks passed.


## 16. 実行報告と保存ファイル確認

In [16]:
baseline_summary = summary_by_model[
    summary_by_model["model_name"]
    == BASELINE_MODEL_NAME
].iloc[0]
external_summary = summary_by_model[
    summary_by_model["model_name"]
    == EXTERNAL_MODEL_NAME
].iloc[0]

transition_counts = (
    comparison["top10_transition"]
    .value_counts()
    .to_dict()
)

run_summary = {
    "notebook_version": NOTEBOOK_VERSION,
    "run_timestamp": datetime.now().isoformat(
        timespec="seconds"
    ),
    "panel_path": str(PANEL_PATH),
    "event_classification_path": str(
        EVENT_CLASSIFICATION_PATH
    ),
    "domain_feature_path": str(DOMAIN_FEATURE_PATH),
    "analysis_start": str(ANALYSIS_START.date()),
    "analysis_end": str(ANALYSIS_END.date()),
    "panel_rows": int(len(panel)),
    "grid_count": int(panel["grid_id"].nunique()),
    "strict_first_occurrence_events": int(
        len(first_events)
    ),
    "event_weeks": int(len(event_groups)),
    "baseline_features": BASE_FEATURES,
    "usable_gis_features": usable_gis_features,
        "excluded_duplicate_gis_features": (
            EXCLUDED_DUPLICATE_GIS_FEATURES
        ),
    "model_parameters": {
        "algorithm": "ExtraTreesClassifier",
        "n_estimators": N_ESTIMATORS,
        "min_samples_leaf": MIN_SAMPLES_LEAF,
        "class_weight": "balanced_subsample",
        "random_state": RANDOM_STATE,
        "negative_sample_ratio": NEGATIVE_SAMPLE_RATIO,
        "minimum_negative_samples": (
            MIN_NEGATIVE_SAMPLES
        ),
        "maximum_negative_samples": (
            MAX_NEGATIVE_SAMPLES
        ),
    },
    "baseline_top10_capture_rate": float(
        baseline_summary["top10_capture_rate"]
    ),
    "external_top10_capture_rate": float(
        external_summary["top10_capture_rate"]
    ),
    "top10_capture_rate_difference": float(
        external_summary["top10_capture_rate"]
        - baseline_summary["top10_capture_rate"]
    ),
    "baseline_top20_capture_rate": float(
        baseline_summary["top20_capture_rate"]
    ),
    "external_top20_capture_rate": float(
        external_summary["top20_capture_rate"]
    ),
    "mean_delta_risk_percentile": float(
        comparison["delta_risk_percentile"].mean()
    ),
    "median_delta_risk_percentile": float(
        comparison["delta_risk_percentile"].median()
    ),
    "top10_transition_counts": transition_counts,
    "top10_exact_p_value": safe_float(exact_p),
    "all_final_checks_passed": bool(
        final_checks["passed"].all()
    ),
}

run_summary_path = (
    AUDIT_DIR / "14_v7_run_summary.json"
)
with open(
    run_summary_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        run_summary,
        file,
        ensure_ascii=False,
        indent=2,
    )

report_lines = [
    "# 14_v7 execution report",
    "",
    f"Generated: {run_summary['run_timestamp']}",
    "",
    "## Inputs",
    f"- Panel: `{PANEL_PATH}`",
    f"- Events: `{EVENT_CLASSIFICATION_PATH}`",
    f"- GIS: `{DOMAIN_FEATURE_PATH}`",
    "",
    "## Evaluation",
    f"- Strict first-occurrence-like events: {len(first_events)}",
    f"- Event weeks: {len(event_groups)}",
    f"- National grids per week: {EXPECTED_GRID_COUNT:,}",
    "",
    "## Feature sets",
    f"- Baseline features: {len(BASE_FEATURES)}",
    f"- External GIS features added: {len(usable_gis_features)}",
    f"- GIS features: {usable_gis_features}",
    "",
    "## Main results",
    (
        "- Baseline Top 10% capture: "
        f"{baseline_summary['top10_capture_rate']:.4f}"
    ),
    (
        "- External Top 10% capture: "
        f"{external_summary['top10_capture_rate']:.4f}"
    ),
    (
        "- Top 10% absolute change: "
        f"{external_summary['top10_capture_rate'] - baseline_summary['top10_capture_rate']:+.4f}"
    ),
    (
        "- Baseline Top 20% capture: "
        f"{baseline_summary['top20_capture_rate']:.4f}"
    ),
    (
        "- External Top 20% capture: "
        f"{external_summary['top20_capture_rate']:.4f}"
    ),
    (
        "- Mean delta risk percentile: "
        f"{comparison['delta_risk_percentile'].mean():+.4f}"
    ),
    (
        "- Median delta risk percentile: "
        f"{comparison['delta_risk_percentile'].median():+.4f}"
    ),
    f"- Top 10 transition counts: {transition_counts}",
    f"- Exact paired transition p-value: {exact_p}",
    "",
    "## Validation",
    (
        "- All final checks passed: "
        f"{bool(final_checks['passed'].all())}"
    ),
]

report_path = (
    RESULT_DIR
    / "14_v7_summary_report_previous_week_weather.md"
)
report_path.write_text(
    "\n".join(report_lines),
    encoding="utf-8",
)

print("Saved:", run_summary_path)
print("Saved:", report_path)
print("\n".join(report_lines))

Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_audit/14_v7_run_summary.json
Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_summary_report_previous_week_weather.md
# 14_v7 execution report

Generated: 2026-06-16T23:41:56

## Inputs
- Panel: `/content/drive/MyDrive/avian_influenza_project/processed/hpai_weekly_grid_panel_with_previous_week_weather_model_ready.parquet`
- Events: `/content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/10_event_occurrence_type_classification.csv`
- GIS: `/content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/18_v8_grid_environment_features_for_14.parquet`

## Evaluation
- Strict first-occurrence-like events: 49
- Event weeks: 35
- National grids per week: 5,491

## Feature sets
- Baseline features: 9
- External GIS features added: 10
- GIS features: ['dist_to_coast_km', 'dist_to_river_km', 'dist_to_lak

In [17]:
expected_outputs = [
    RESULT_DIR / "14_v7_strict_first_occurrence_events_used.csv",
    RESULT_DIR / "14_v7_domain_feature_source.csv",
    RESULT_DIR / "14_v7_domain_features_used_whitelist.csv",
    RESULT_DIR / "14_v7_feature_sets_previous_week_weather.csv",
    RESULT_DIR / "14_v7_shared_training_sample_manifest.csv",
    RESULT_DIR / "14_v7_case_results_previous_week_weather_same_extratrees.csv",
    RESULT_DIR / "14_v7_training_log_previous_week_weather_same_extratrees.csv",
    RESULT_DIR / "14_v7_all_grid_predictions_event_weeks.parquet",
    RESULT_DIR / "14_v7_summary_by_model_previous_week_weather.csv",
    RESULT_DIR / "14_v7_summary_by_fiscal_year_model_previous_week_weather.csv",
    RESULT_DIR / "14_v7_case_level_baseline_vs_external_previous_week_weather.csv",
    RESULT_DIR / "14_v7_overall_baseline_vs_external_previous_week_weather.csv",
    RESULT_DIR / "14_v7_gain_loss_category_summary_previous_week_weather.csv",
    RESULT_DIR / "14_v7_gain_loss_summary_by_fiscal_year_previous_week_weather.csv",
    RESULT_DIR / "14_v7_paired_bootstrap_summary.csv",
    RESULT_DIR / "14_v7_top10_transition_exact_test.csv",
    RESULT_DIR / "14_v7_map_ready_external_gis_gain_loss_cases.csv",
    RESULT_DIR / "14_v7_case_level_gain_loss_enriched.csv",
    RESULT_DIR / "14_v7_baseline_top10_success_miss_cases.csv",
    RESULT_DIR / "14_v7_external_top10_success_miss_cases.csv",
    RESULT_DIR / "14_v7_feature_importance_summary.csv",
    RESULT_DIR / "14_v7_summary_report_previous_week_weather.md",
    AUDIT_DIR / "14_v7_panel_validation.csv",
    AUDIT_DIR / "14_v7_event_validation.csv",
    AUDIT_DIR / "14_v7_gis_whitelist_inventory.csv",
    AUDIT_DIR / "14_v7_gis_pairwise_duplicate_check.csv",
    AUDIT_DIR / "14_v7_excluded_duplicate_gis_features.csv",
    AUDIT_DIR / "14_v7_selected_feature_audit.csv",
    AUDIT_DIR / "14_v7_final_validation_checks.csv",
    AUDIT_DIR / "14_v7_run_summary.json",
]

output_inventory = pd.DataFrame(
    [
        {
            "file": path.name,
            "path": str(path),
            "exists": path.exists(),
            "size_bytes": (
                int(path.stat().st_size)
                if path.exists()
                else 0
            ),
            "modified_at": (
                datetime.fromtimestamp(
                    path.stat().st_mtime
                ).isoformat(timespec="seconds")
                if path.exists()
                else None
            ),
        }
        for path in expected_outputs
    ]
)

save_audit_csv(
    output_inventory,
    "14_v7_output_file_inventory.csv",
)
display(output_inventory)

missing_outputs = output_inventory[
    ~output_inventory["exists"]
]
if not missing_outputs.empty:
    display(missing_outputs)
    raise FileNotFoundError(
        "14_v7の予定出力の一部が保存されていません．"
    )

print("All expected 14_v7 outputs were saved.")

Saved: /content/drive/MyDrive/avian_influenza_project/processed/model_outputs_riskmap_eval/14_v7_audit/14_v7_output_file_inventory.csv (30, 5)


,file,path,exists,size_bytes,modified_at
0,14_v7_strict_first_occurrence_events_used.csv,/content/drive/MyDrive/avian_influenza_project...,True,1210,2026-06-16T23:39:40
1,14_v7_domain_feature_source.csv,/content/drive/MyDrive/avian_influenza_project...,True,154,2026-06-16T23:39:50
2,14_v7_domain_features_used_whitelist.csv,/content/drive/MyDrive/avian_influenza_project...,True,191,2026-06-16T23:39:50
3,14_v7_feature_sets_previous_week_weather.csv,/content/drive/MyDrive/avian_influenza_project...,True,532,2026-06-16T23:39:51
4,14_v7_shared_training_sample_manifest.csv,/content/drive/MyDrive/avian_influenza_project...,True,4518,2026-06-16T23:40:18
5,14_v7_case_results_previous_week_weather_same_...,/content/drive/MyDrive/avian_influenza_project...,True,26280,2026-06-16T23:41:53
6,14_v7_training_log_previous_week_weather_same_...,/content/drive/MyDrive/avian_influenza_project...,True,11320,2026-06-16T23:41:53
7,14_v7_all_grid_predictions_event_weeks.parquet,/content/drive/MyDrive/avian_influenza_project...,True,5212619,2026-06-16T23:41:53
8,14_v7_summary_by_model_previous_week_weather.csv,/content/drive/MyDrive/avian_influenza_project...,True,831,2026-06-16T23:41:54
9,14_v7_summary_by_fiscal_year_model_previous_we...,/content/drive/MyDrive/avian_influenza_project...,True,1745,2026-06-16T23:41:54


All expected 14_v7 outputs were saved.


## 次の判断

実行後は，まず以下の3ファイルを確認する．

1. `14_v7_summary_by_model_previous_week_weather.csv`
2. `14_v7_gain_loss_category_summary_previous_week_weather.csv`
3. `14_v7_final_validation_checks.csv`

主結果が確定した後に，Figure 3–5，Table 3–6，Abstract，Results，
Discussion，Conclusionを14_v7出力へ統一する．